In [22]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_PATH = Path("C:/Users/Архиповы/source/repos/modem")
sys.path.insert(0, str(PROJECT_PATH / "src"))

from modem.blocks import MessageSource, StringToBits, PacketBuileder, PCM, BPSK, HammingDecoder, HammongEncoder
from modem.blocks import BFSK, Channel, BFSKDemodulator, PacketDecoder, Demodulator, BitsToString
from modem.pipeline import run

MAX_DISPLAY_SAMPLES = 2000
MAX_DISPLAY_BITS = 30

def modulation_graph(mod_signal, samp_rate, name, ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
    n_show = min(len(mod_signal), MAX_DISPLAY_SAMPLES)
    t = np.arange(n_show) / samp_rate
    ax.plot(t, mod_signal[:n_show], color='blue')
    ax.set_xlabel("t, сек")
    ax.set_ylabel("Ампл")
    ax.grid(True, alpha=0.3)
    ax.set_title(f"{name} сигнал")
    return ax

def channel_graph(input_signal, output_signal, samp_rate, ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
    n_show = min(len(input_signal), MAX_DISPLAY_SAMPLES)
    t = np.arange(n_show) / samp_rate
    ax.plot(t, input_signal[:n_show], label='Вход', color='blue')
    ax.plot(t, output_signal[:n_show], label='Выход', color='red')
    ax.set_xlabel("t, сек")
    ax.set_ylabel("Ампл")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title("Канал")
    return ax

def calculate_ber(original_bits, recovered_bits):
    min_len = min(len(original_bits), len(recovered_bits))
    if min_len == 0:
        return 1.0
    errors = np.sum(original_bits[:min_len] != recovered_bits[:min_len])
    return errors / min_len

mod_type = widgets.RadioButtons(options=['BPSK', 'BFSK'], value='BPSK', description='Модуляция:')
fc_slider = widgets.IntSlider(value=300, min=50, max=2000, step=50, description='Fc (Гц):')
Rb_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='R_b (бит/с):')
h_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='h (BFSK):')
k_slider = widgets.FloatSlider(value=0.95, min=0.1, max=1.0, step=0.01, description='Затухание:')
snr_slider = widgets.FloatSlider(value=30, min=-20, max=50, step=1, description='SNR (дБ):')
noise_type = widgets.Dropdown(options=['gaussian', 'laplace', 'none'], value='gaussian', description='Тип шума:')
phase_noise_slider = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description='Фазовый шум:')
use_doppler = widgets.Checkbox(value=False, description='Эффект Доплера')
velocity_slider = widgets.FloatSlider(value=30.0, min=0.0, max=100.0, step=1.0, description='Скорость (м/с):')
angle_slider = widgets.FloatSlider(value=0.0, min=0.0, max=3.14, step=0.1, description='Угол (рад):')
use_hamming = widgets.Checkbox(value=True, description='Код Хэмминга')

source_type = widgets.RadioButtons(options=[('Ввести текст', 'text'), ('Загрузить из файла', 'file')], value='text', description='Источник:')
text_input = widgets.Textarea(value='Hello DSP!', description='Текст:', layout=widgets.Layout(width='100%', height='80px'))
file_input = widgets.Text(value='message.txt', description='Файл:', disabled=True)

def update_source(*args):
    file_input.disabled = (source_type.value != 'file')
source_type.observe(update_source, 'value')

run_button = widgets.Button(description='Старт', button_style='success', layout=widgets.Layout(width='200px'))
output = widgets.Output()

ui = widgets.VBox([
    widgets.HTML("<h2>Настройки системы связи</h2>"),
    mod_type, fc_slider, Rb_slider, h_slider,
    widgets.HTML("<hr>"),
    k_slider, snr_slider, noise_type, phase_noise_slider,
    use_doppler, velocity_slider, angle_slider,
    use_hamming,
    widgets.HTML("<hr>"),
    source_type, text_input, file_input,
    widgets.HTML("<hr>"),
    run_button, output
])

display(ui)

def run_simulation(b):
    with output:
        clear_output(wait=True)
        
        mod_type_val = mod_type.value
        fc = fc_slider.value
        R_b = Rb_slider.value
        k = k_slider.value
        snr_db = snr_slider.value
        noise_type_val = noise_type.value
        phase_noise_std = phase_noise_slider.value
        use_doppler_val = use_doppler.value
        velocity = velocity_slider.value
        angle = angle_slider.value
        use_hamming_val = use_hamming.value
        h = h_slider.value
        
        samp_rate = fc * 20
        samp_per_bit = int(samp_rate / R_b)
        f0 = fc - R_b * h
        f1 = fc + R_b * h
        
        if source_type.value == 'text':
            text = text_input.value
            filepath = None
        else:
            filepath = file_input.value
            text = None
            if not Path(filepath).exists():
                print(f"Файл {filepath} не найден")
                return
        
        ctx = {}
        capture = {}
        
        blocks = [
            MessageSource(filepath=filepath, default_text=text or "Hello", name="MS"),
            StringToBits(encoding='utf-8', name="StB"),
        ]
        
        if use_hamming_val:
            blocks.append(HammongEncoder(name="HE"))
        
        blocks.append(PacketBuileder(preamble="1010101010101010", sfd="10101011", add_crc=True, name="packet"))
        blocks.append(PCM(name="PCM"))
        
        if mod_type_val == "BPSK":
            blocks.append(BPSK(freq_carrier=fc, R_b=R_b, name="bpsk"))
        else:
            blocks.append(BFSK(freq_carrier=fc, R_b=R_b, h=h, name="BFSK"))
        
        blocks.append(Channel(
            k=k, snr_db=snr_db, noise_type=noise_type_val,
            phase_noise_std=phase_noise_std,
            use_doppler=use_doppler_val,
            velocity=velocity, angle=angle,
            fc=fc, samp_rate=samp_rate,
            name="channel"
        ))
        
        if mod_type_val == "BPSK":
            blocks.append(Demodulator(freq_carrier=fc, samp_rate=samp_rate, R_b=R_b, lpf_cutoff_multiplier=1.5, name="demod"))
        else:
            blocks.append(BFSKDemodulator(f0=f0, f1=f1, samp_rate=samp_rate, samp_per_bit=samp_per_bit, name="BFSKDM"))
        
        blocks.append(PacketDecoder(preamble="1010101010101010", sfd="10101011", name="PackDec"))
        
        if use_hamming_val:
            blocks.append(HammingDecoder(name="HammDecoder"))
        
        blocks.append(BitsToString(encoding='utf-8', name="BSt"))
        
        result = run(blocks, x=None, ctx=ctx, capture=capture)
        
        original_text = ctx.get("source_message", capture.get("MS", "???"))
        decoded_text = ctx.get("decoded_text", capture.get("BSt", "???"))
        original_bits = ctx.get("bits", capture.get("StB", []))
        
        if mod_type_val == "BPSK":
            recovered_bits = ctx.get("bpsk_demod", capture.get("demod", []))
            mod_signal = ctx.get("BPSK_signal", capture.get("bpsk", []))
            before_lpf = ctx.get("before_LPF", [])
            after_lpf = ctx.get("after_LPF", [])
        else:
            recovered_bits = ctx.get("bfsk_demod", capture.get("BFSKDM", []))
            mod_signal = ctx.get("BFSK_signal_good", capture.get("BFSK", []))
            envelope0 = ctx.get("bfsk_envelope0", [])
            envelope1 = ctx.get("bfsk_envelope1", [])
        
        channel_out = ctx.get("channel_output", capture.get("channel", []))
        ber = calculate_ber(original_bits, recovered_bits)
        
        print(f"\nОтправлено: {original_text}")
        print(f"Получено: {decoded_text}")
        print(f"BER: {ber:.8f}")
        
        if original_text == decoded_text:
            print("Сообщение передано без ошибок\n")
        else:
            print("Ошибки при передаче\n")
        
        if len(mod_signal) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            modulation_graph(mod_signal, samp_rate, mod_type_val, ax)
            plt.tight_layout()
            plt.show()
        
        if len(mod_signal) > 0 and len(channel_out) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            channel_graph(mod_signal, channel_out, samp_rate, ax)
            plt.tight_layout()
            plt.show()
        
        if len(recovered_bits) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            n_show = min(len(recovered_bits), MAX_DISPLAY_BITS)
            t_bits = np.arange(n_show) * (samp_per_bit / samp_rate)
            ax.step(t_bits, recovered_bits[:n_show], where='post', color='red')
            ax.set_xlabel("t, сек")
            ax.set_ylabel("Ампл")
            ax.grid(True, alpha=0.3)
            ax.set_title("Демодулированные биты")
            ax.set_ylim(-0.2, 1.2)
            plt.tight_layout()
            plt.show()
        
        if mod_type_val == "BPSK" and len(before_lpf) > 0 and len(after_lpf) > 0:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
            n_show = min(len(before_lpf), 1000)
            t = np.arange(n_show) / samp_rate
            ax1.plot(t, before_lpf[:n_show], color='blue')
            ax1.set_title("Сигнал до ФНЧ")
            ax1.grid(True)
            ax2.plot(t, after_lpf[:n_show], color='red')
            ax2.set_title("Сигнал после ФНЧ")
            ax2.set_xlabel("t, сек")
            ax2.grid(True)
            plt.tight_layout()
            plt.show()
        
        if mod_type_val == "BFSK" and len(envelope0) > 0 and len(envelope1) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            n_show = min(len(envelope0), MAX_DISPLAY_SAMPLES)
            t = np.arange(n_show) / samp_rate
            ax.plot(t, envelope0[:n_show], label='Огибающая f0', color='blue')
            ax.plot(t, envelope1[:n_show], label='Огибающая f1', color='red')
            ax.set_xlabel("t, сек")
            ax.set_ylabel("Ампл")
            ax.grid(True, alpha=0.3)
            ax.legend()
            ax.set_title("Огибающие BFSK")
            plt.tight_layout()
            plt.show()

run_button.on_click(run_simulation)